# Análise de Reembolso de Despesas

Este notebook demonstra como criar agentes que utilizam plugins para processar despesas de viagem a partir de imagens locais de recibos, gerar um e-mail de solicitação de reembolso e visualizar dados de despesas usando um gráfico de pizza. Os agentes escolhem funções dinamicamente com base no contexto da tarefa.

Passos:
1. O Agente OCR processa a imagem local do recibo e extrai os dados de despesas de viagem.
2. O Agente de E-mail gera um e-mail de solicitação de reembolso.

### Exemplo de um cenário de despesa de viagem:
Imagine que você é um funcionário viajando para uma reunião de negócios em outra cidade. Sua empresa tem uma política de reembolso para todas as despesas razoáveis relacionadas à viagem. Aqui está a divisão das possíveis despesas de viagem:
- Transporte:
Passagem aérea para uma viagem de ida e volta da sua cidade de origem até a cidade de destino.
Táxi ou serviços de transporte por aplicativo de ida e volta ao aeroporto.
Transporte local na cidade de destino (como transporte público, carros de aluguel ou táxis).

- Hospedagem:
Estadia em hotel por três noites em um hotel de negócios de médio padrão próximo ao local da reunião.

- Refeições:
Diária de alimentação para café da manhã, almoço e jantar, baseada na política de per diem da empresa.

- Despesas Diversas:
Taxas de estacionamento no aeroporto.
Cobranças de acesso à internet no hotel.
Gorjetas ou pequenas taxas de serviço.

- Documentação:
Você envia todos os recibos (voos, táxis, hotel, refeições, etc.) e um relatório de despesas preenchido para reembolso.


## Importar bibliotecas necessárias

Importe as bibliotecas e módulos necessários para o notebook.


In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import base64
import dotenv
from typing import Annotated, List

from pydantic import BaseModel, Field

from agent_framework import tool, AgentResponseUpdate, WorkflowBuilder
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Azure AI Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

 ## Definir Modelos de Despesa

 Crie um modelo Pydantic para despesas individuais e uma classe ExpenseFormatter para converter uma consulta do usuário em dados estruturados de despesa.

 Cada despesa será representada no formato:
 `{'date': '07-Mar-2025', 'description': 'flight to destination', 'amount': 675.99, 'category': 'Transportation'}`


In [ ]:
class Expense(BaseModel):
    date: str = Field(..., description="Date of expense in dd-MMM-yyyy format")
    description: str = Field(..., description="Expense description")
    amount: float = Field(..., description="Expense amount")
    category: str = Field(..., description="Expense category (e.g., Transportation, Meals, Accommodation, Miscellaneous)")

class ExpenseFormatter(BaseModel):
    raw_query: str = Field(..., description="Raw query input containing expense details")
    
    def parse_expenses(self) -> List[Expense]:
        """
        Parses the raw query into a list of Expense objects.
        Expected format: "date|description|amount|category" separated by semicolons.
        """
        expense_list = []
        for expense_str in self.raw_query.split(";"):
            if expense_str.strip():
                parts = expense_str.strip().split("|")
                if len(parts) == 4:
                    date, description, amount, category = parts
                    try:
                        expense = Expense(
                            date=date.strip(),
                            description=description.strip(),
                            amount=float(amount.strip()),
                            category=category.strip()
                        )
                        expense_list.append(expense)
                    except ValueError as e:
                        print(f"[LOG] Parse Error: Invalid data in '{expense_str}': {e}")
        return expense_list

## Definindo Ferramentas - Gerando o Email

Crie uma função de ferramenta para gerar um email para enviar uma solicitação de reembolso de despesas.
- Esta ferramenta usa o decorador `@tool` do Microsoft Agent Framework.
- Ela calcula o valor total das despesas e formata os detalhes no corpo do email.


In [ ]:
@tool(approval_mode="never_require")
def generate_expense_email(
    expense_data: Annotated[str, "Semicolon-separated expense entries in 'date|description|amount|category' format"]
) -> str:
    """Generate an email to submit an expense claim to the Finance Team."""
    formatter = ExpenseFormatter(raw_query=expense_data)
    expenses = formatter.parse_expenses()
    if not expenses:
        return "No valid expenses found to include in the email."
    total_amount = sum(e.amount for e in expenses)
    email_body = "Dear Finance Team,\n\n"
    email_body += "Please find below the details of my expense claim:\n\n"
    for e in expenses:
        email_body += f"- {e.date} | {e.description}: ${e.amount:.2f} ({e.category})\n"
    email_body += f"\nTotal Amount: ${total_amount:.2f}\n\n"
    email_body += "Receipts for all expenses are attached for your reference.\n\n"
    email_body += "Thank you,\n[Your Name]"
    return email_body

# Ferramenta para Extração de Despesas de Viagem de Imagens de Recibos

Crie uma função de ferramenta para extrair despesas de viagem de imagens de recibos.
- Esta ferramenta usa o decorador `@tool` do Microsoft Agent Framework.
- Ela lê a imagem do recibo, codifica-a em base64 e retorna o URI de dados para o agente analisar.


In [ ]:
@tool(approval_mode="never_require")
def load_receipt_image(
    image_path: Annotated[str, "Path to the receipt image file"] = "receipt.jpg"
) -> str:
    """Load a receipt image and return its base64-encoded data URI for OCR extraction."""
    try:
        with open(image_path, "rb") as f:
            image_data = base64.b64encode(f.read()).decode("utf-8")
        return f"data:image/jpeg;base64,{image_data}"
    except Exception as e:
        error_msg = f"[LOG] Error loading image '{image_path}': {str(e)}"
        print(error_msg)
        return error_msg

## Processando Despesas

Defina os agentes e conecte-os em um fluxo de trabalho sequencial usando `WorkflowBuilder`.
- O agente OCR extrai dados estruturados de despesas da imagem do recibo usando a ferramenta `load_receipt_image`.
- O agente de Email pega os dados extraídos e gera um e-mail profissional de solicitação de reembolso usando a ferramenta `generate_expense_email`.
- `WorkflowBuilder` com `add_edge` cria um pipeline sequencial: Agente OCR → Agente de Email.


In [ ]:
ocr_agent = client.as_agent(
    tools=[load_receipt_image],
    name="OCRAgent",
    instructions=(
        "You are an expert OCR assistant specialized in extracting structured data from receipt images. "
        "Use the 'load_receipt_image' tool to load the receipt image, then analyze it and extract "
        "travel-related expense details in the format: 'date|description|amount|category' separated by semicolons. "
        "Follow these rules: "
        "- Date: Convert dates (e.g., '4/4/22') to 'dd-MMM-yyyy' (e.g., '04-Apr-2022'). "
        "- Description: Extract item names. "
        "- Amount: Use numeric values (e.g., '4.50' from '$4.50'). "
        "- Category: Infer from context (e.g., 'Meals' for food, 'Transportation' for travel, "
        "'Accommodation' for lodging, 'Miscellaneous' otherwise). "
        "Ignore totals, subtotals, or service charges unless they are itemized expenses. "
        "If no expenses are found, return 'No expenses detected'. "
        "Return only the structured data, no additional text."
    ),
)

email_agent = client.as_agent(
    name="EmailAgent",
    tools=[generate_expense_email],
    instructions=(
        "You are an expense claim email generator. Take the travel expense data from the previous agent "
        "(in 'date|description|amount|category' format separated by semicolons) and use the "
        "'generate_expense_email' tool to produce a professional expense claim email. "
        "Pass the semicolon-separated expense data directly to the tool."
    ),
)

## Função principal

Construa o fluxo de trabalho sequencial e execute-o para processar a imagem do recibo e gerar o e-mail de reembolso de despesas.


> **Nota:** Este fluxo de trabalho atualmente passa a imagem do recibo como texto base64, que a maioria dos modelos de chat (incluindo o gpt-4o) não tratará como uma imagem.  
> Também pode exceder a janela de contexto do modelo. Prefira executar OCR com Azure AI Vision (ou outra ferramenta de OCR) e passe apenas o texto extraído, ou refatore para enviar a imagem como uma mensagem `image_url`.  
> Se você quiser apenas evitar erros de contexto, tente uma imagem de recibo menor ou um modelo com uma janela de contexto maior.


In [ ]:
workflow = WorkflowBuilder(start_executor=ocr_agent) \
    .add_edge(ocr_agent, email_agent) \
    .build()

prompt = (
    "Please extract the raw text from the receipt image at 'receipt.jpg', "
    "focusing on travel expenses like dates, descriptions, amounts, and categories "
    "(e.g., Transportation, Accommodation, Meals, Miscellaneous). "
    "Then generate a professional expense claim email."
)

last_author = None
events = workflow.run(
    prompt,
    stream=True,
)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"# Agent - {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Aviso Legal**:
Este documento foi traduzido usando o serviço de tradução por IA [Co-op Translator](https://github.com/Azure/co-op-translator). Embora nos esforcemos pela precisão, por favor, esteja ciente de que traduções automatizadas podem conter erros ou imprecisões. O documento original em seu idioma nativo deve ser considerado a fonte autorizada. Para informações críticas, recomenda-se tradução profissional humana. Não nos responsabilizamos por quaisquer mal-entendidos ou interpretações incorretas decorrentes do uso desta tradução.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
